# LMA Phase 2 BONUS: Bhojpuri Transformer Pretraining -- NO POSITIONAL EMBEDDINGS Ablation

(LMA_Individual_Project_v1.pdf, "Bonus (optional): Ablation -- no positional embeddings")

New notebook -- does not modify pretrain_kaggle.ipynb or any existing code. Trains
`BhojpuriTransformerNoPos` (bhojpuri/model/transformer_no_pos.py, a subclass of the standard
`BhojpuriTransformer` with the positional-embedding table removed) via a new entry point
(`train/train_no_pos.py`) that reuses the real `Trainer`/`load_config`/`PackedLMDataset`/
`BhojpuriTokenizer` classes unchanged -- only the model class and its config file differ from
the standard pretraining run. Same raw corpus, same hyperparameters (configs/training_config.json)
as the standard high-parameter Bhojpuri run -- the only variable changed is the presence/absence
of positional embeddings, isolating that one ablation factor. No synthetic data.


In [ ]:
# ===== CONFIG-ONLY CELL: Modify these before running =====
ROOT_DIR = "/kaggle/input/lma-bhojpuri-phase2"  # Code+config bundle path -- MUST be re-uploaded to include the new train/train_no_pos.py, model/transformer_no_pos.py, configs/model_config_no_pos.json files (dataset-metadata.json id: kspsvln/lma-bhojpuri-phase2)
DATA_DIR = "/kaggle/input/<your-bhojpuri-data-dataset>/data"  # Same raw corpus as the standard run: expects train/bhoj.txt, val/bhoj.txt, test/bhoj.txt under this dir -- no synthetic data
OUT_DIR = "/kaggle/working/checkpoints_no_pos"   # Distinct from the standard run's OUT_DIR so the two checkpoints never collide
CHECK_DIR = None                          # Resume from a prior no-pos checkpoint (if available)

LANGUAGE = "bhojpuri"

# Hyperparameters: None means use config JSON defaults (configs/training_config.json) -- same
# hyperparameters as the standard Bhojpuri pretraining run, so the only ablated variable is the
# model architecture (positional embeddings removed), not the training recipe.
hp = dict(
    batch_size=None,
    learning_rate=None,
    num_epochs=None,
    warmup_steps=None,
    weight_decay=None,
    max_seq_len=None,
    amp=True,
)


In [ ]:
import sys
import os
import argparse
from pathlib import Path

# Fail fast if ROOT_DIR isn't actually where the bundle is mounted -- sys.path.insert()
# silently accepts a bad path, so a wrong ROOT_DIR would otherwise only surface later as a
# confusing "ModuleNotFoundError: No module named 'train'" at the import cell.
root_path = Path(ROOT_DIR)
expected_entry = root_path / "train" / "train_no_pos.py"
if not expected_entry.exists():
    kaggle_input = Path("/kaggle/input")
    available = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    listing = "\n".join(f"  {p}" for p in sorted(root_path.iterdir())) if root_path.exists() else "  (ROOT_DIR does not exist)"
    raise RuntimeError(
        f"Expected {expected_entry} but it's not there.\n"
        f"ROOT_DIR = {ROOT_DIR}\n"
        f"Contents of ROOT_DIR:\n{listing}\n"
        f"Datasets attached under /kaggle/input/: {available}\n"
        "Check that the lma-bhojpuri-phase2 bundle was RE-UPLOADED to include the new "
        "train/train_no_pos.py, model/transformer_no_pos.py, configs/model_config_no_pos.json "
        "files (this bonus ablation needs a newer bundle version than the standard run does), "
        "that it's attached as an input to this notebook (Add Input), and that ROOT_DIR above "
        "matches its actual mounted path -- it may be nested one level deeper "
        "(e.g. /kaggle/input/<slug>/kaggle_bundle/) depending on how it was uploaded."
    )

# Setup path to import from bundle
sys.path.insert(0, ROOT_DIR)
os.chdir('/kaggle/working')  # For checkpoint/log output

# Install tokenizers if needed
import subprocess
subprocess.run(['pip', 'install', 'tokenizers'], capture_output=True)

print(f'Root: {ROOT_DIR}')
print(f'Output: {OUT_DIR}')
print(f'Resume: {CHECK_DIR}')


In [ ]:
# Import the real training code (no reimplementation) -- the NO-POS ablation entry point
from train.train_no_pos import run_training_no_pos

print('✅ Imported no-positional-embeddings ablation training code from bundle')


In [ ]:
# Build command-line arguments by mimicking train_no_pos.py's argparse
# Filter out None hyperparams (use config JSON defaults)
args_dict = {k: v for k, v in hp.items() if v is not None}

# Handle resume: compute resume_from path
resume_from = None
if CHECK_DIR and Path(CHECK_DIR).exists():
    potential_ckpt = Path(CHECK_DIR) / 'checkpoint_last.pt'
    if potential_ckpt.exists():
        resume_from = str(potential_ckpt)
        print(f'Will resume from: {resume_from}')

# Create args namespace (fields must match train_no_pos.py's run_training_no_pos() override_config exactly)
args = argparse.Namespace(
    batch_size=args_dict.get('batch_size', None),
    learning_rate=args_dict.get('learning_rate', None),
    num_epochs=args_dict.get('num_epochs', None),
    warmup_steps=args_dict.get('warmup_steps', None),
    weight_decay=args_dict.get('weight_decay', None),
    max_seq_len=args_dict.get('max_seq_len', None),
    amp=args_dict.get('amp', True),
    resume_from=resume_from,
)

print('✅ Arguments prepared')


In [ ]:
# Run training with the real Trainer class (checkpointing, logging, AMP, etc. all included),
# but with BhojpuriTransformerNoPos as the model class instead of the standard BhojpuriTransformer
print('\n' + '='*60)
print(f'Starting {LANGUAGE.upper()} NO-POSITIONAL-EMBEDDINGS ablation training...')
print('='*60 + '\n')

run_training_no_pos(args, root_dir=ROOT_DIR, data_dir=DATA_DIR, output_dir=OUT_DIR)

print('\n' + '='*60)
print('✅ Training complete!')
print('='*60)


In [ ]:
# Final summary
import glob

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint_*.pt'))
logs = sorted(glob.glob('logs_no_pos/*.log'))

print(f'\n📊 Training outputs (no-positional-embeddings ablation):')
if checkpoints:
    print(f'  Checkpoints:')
    for ckpt in checkpoints:
        size_mb = Path(ckpt).stat().st_size / 1e6
        print(f'    {ckpt} ({size_mb:.1f} MB)')

if logs:
    print(f'  Logs:')
    for log in logs:
        print(f'    {log}')

print(f'\n📝 To resume in next run:')
print(f'  1. Save this notebook output as a Kaggle dataset')
print(f'  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/checkpoints_no_pos"')
print(f'  3. Run the notebook again')


In [ ]:
# ============================================================================
# Phase 2 evaluation metrics on the held-out test set (LMA_Individual_Project_v1.pdf, Sec 2.3),
# run against the NO-POSITIONAL-EMBEDDINGS ablation checkpoint -- same metric suite as the
# standard pretrain_kaggle.ipynb eval cell, so results are directly comparable:
#   [1] Perplexity / bits-per-byte -- full test set
#   [2] Generation quality: greedy + temperatures [0.5, 1.0, 1.5] vs. reference continuations
#       -> BLEU-4, chrF++, ROUGE-L, repetition rate, Distinct-1/2 (on generated text)
#   [3] Attention analysis: entropy + mean attention distance PER HEAD (first & last layer),
#       plus heatmap plots (first & last layer, all heads) for an example sentence
# ============================================================================
import subprocess
subprocess.run(['pip', 'install', '-q', 'sacrebleu', 'rouge_score'], capture_output=True)

import json
import math
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sacrebleu
from rouge_score import rouge_scorer

from model.transformer_no_pos import BhojpuriTransformerNoPos as ModelClass
from tokenizer.tokenizer_wrapper import BhojpuriTokenizer as TokenizerClass
test_filename = "bhoj.txt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best checkpoint (fall back to last)
ckpt_path = Path(OUT_DIR) / "checkpoint_best.pt"
if not ckpt_path.exists():
    ckpt_path = Path(OUT_DIR) / "checkpoint_last.pt"
assert ckpt_path.exists(), f"No checkpoint found in {OUT_DIR}"

tokenizer = TokenizerClass()
print(f"✓ Loaded tokenizer (vocab_size={tokenizer.vocab_size})")

model_config_path = Path(ROOT_DIR) / "configs" / "model_config_no_pos.json"
model = ModelClass(config_path=str(model_config_path), vocab_size=tokenizer.vocab_size).to(device)
checkpoint = torch.load(ckpt_path, map_location=device)
state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()
print(f"✓ Loaded {ckpt_path.name} ({model.count_parameters():,} params, no positional embeddings)")

test_txt = Path(DATA_DIR) / "test" / test_filename
assert test_txt.exists(), f"Test data not found at {test_txt}"
with open(test_txt, "r", encoding="utf-8") as f:
    test_texts = [l.strip() for l in f if l.strip()]
print(f"✓ Loaded {len(test_texts):,} test lines from {test_txt}")

results = {"language": LANGUAGE, "ablation": "no_positional_embeddings", "checkpoint": str(ckpt_path)}

# --- [1/3] Perplexity & bits-per-byte (full test set) ---
print(f"\n[1/3] Perplexity & bits-per-byte (full test set, {len(test_texts):,} lines)...")
total_loss, total_tokens, n_samples = 0.0, 0, 0
with torch.no_grad():
    for i, text in enumerate(test_texts):
        ids = tokenizer.encode(text, add_special_tokens=False)
        if len(ids) < 2:
            continue
        ids = ids[:model.max_seq_len]  # cap to the trained context window
        input_ids = torch.tensor([ids[:-1]], device=device)
        target_ids = torch.tensor(ids[1:], device=device)
        logits, _ = model(input_ids)
        loss = F.cross_entropy(logits[0, :len(target_ids)], target_ids)
        total_loss += loss.item() * len(target_ids)
        total_tokens += len(target_ids)
        n_samples += 1
        if (i + 1) % 20000 == 0:
            print(f"   ...{i + 1:,}/{len(test_texts):,} lines processed")

avg_loss = total_loss / total_tokens if total_tokens else float("nan")
ppl = math.exp(min(avg_loss, 20))
bpb = avg_loss / math.log(2)
results["perplexity"] = {
    "perplexity": round(ppl, 2),
    "cross_entropy_loss": round(avg_loss, 4),
    "bits_per_byte": round(bpb, 4),
    "samples_evaluated": n_samples,
    "total_tokens": total_tokens,
}
print(f"   PPL={ppl:.2f}  BPB={bpb:.4f}  samples={n_samples}  tokens={total_tokens:,}")

# --- [2/3] Generation quality: greedy + temperatures [0.5, 1.0, 1.5] vs. reference continuations ---
print("\n[2/3] Generation quality vs. reference continuations (greedy + temperatures 0.5/1.0/1.5)...")

PREFIX_CHARS, REF_CHARS, NUM_GEN_SAMPLES = 50, 100, 1000
gen_pairs = []
for text in test_texts:
    if len(text) < PREFIX_CHARS + 10:
        continue
    prefix, reference = text[:PREFIX_CHARS], text[PREFIX_CHARS:PREFIX_CHARS + REF_CHARS]
    if reference.strip():
        gen_pairs.append((prefix, reference))
    if len(gen_pairs) >= NUM_GEN_SAMPLES:
        break
print(f"   Using {len(gen_pairs)} held-out (prefix, reference) pairs")

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def repetition_rate(texts, n=2):
    """Fraction of n-grams that are repeats within their own text (lower = less repetitive)."""
    total, repeated = 0, 0
    for t in texts:
        toks = t.split()
        grams = list(zip(*[toks[i:] for i in range(n)]))
        if not grams:
            continue
        seen = Counter(grams)
        total += len(grams)
        repeated += sum(c - 1 for c in seen.values() if c > 1)
    return round(repeated / total, 4) if total else 0.0

def distinct_n(texts, n=1):
    grams = Counter()
    total = 0
    for t in texts:
        toks = t.split()
        g = list(zip(*[toks[i:] for i in range(n)])) if n > 1 else toks
        grams.update(g)
        total += len(g)
    return round(len(grams) / total, 4) if total else 0.0

gen_results = {}
decoding_modes = [("greedy", None, True), ("temp_0.5", 0.5, False), ("temp_1.0", 1.0, False), ("temp_1.5", 1.5, False)]

for mode_name, temp, greedy in decoding_modes:
    hyps, refs = [], []
    for i, (prefix, reference) in enumerate(gen_pairs):
        ids = tokenizer.encode(prefix, add_special_tokens=False)
        if not ids:
            continue
        ref_ids = tokenizer.encode(reference, add_special_tokens=False)
        n_new = max(5, min(len(ref_ids), 40))
        input_ids = torch.tensor([ids], device=device)
        with torch.no_grad():
            generated = model.generate(input_ids, max_new_tokens=n_new, temperature=(temp or 1.0), greedy=greedy)
        gen_ids = generated[0].cpu().numpy().tolist()[len(ids):]
        hyps.append(tokenizer.decode(gen_ids))
        refs.append(reference)
        if (i + 1) % 200 == 0:
            print(f"   [{mode_name}] ...{i + 1}/{len(gen_pairs)} prompts generated")

    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score if hyps else 0.0
    chrf_pp = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score if hyps else 0.0  # chrF++
    rouge_l = float(np.mean([rouge.score(r, h)['rougeL'].fmeasure for h, r in zip(hyps, refs)])) if hyps else 0.0

    gen_results[mode_name] = {
        "num_samples": len(hyps),
        "bleu_4": round(bleu, 2),
        "chrf++": round(chrf_pp, 2),
        "rouge_l_f1": round(rouge_l, 4),
        "repetition_rate_bigram": repetition_rate(hyps, n=2),
        "distinct_1": distinct_n(hyps, n=1),
        "distinct_2": distinct_n(hyps, n=2),
        "examples": [{"prefix": p, "reference": r, "generated": h} for (p, r), h in list(zip(gen_pairs, hyps))[:3]],
    }
    r = gen_results[mode_name]
    print(f"   {mode_name}: BLEU-4={r['bleu_4']}  chrF++={r['chrf++']}  ROUGE-L={r['rouge_l_f1']}  "
          f"rep_rate={r['repetition_rate_bigram']}  D1={r['distinct_1']}  D2={r['distinct_2']}  n={r['num_samples']}")

results["generation"] = gen_results

# --- [3/3] Attention analysis: entropy + mean attention distance per head, first & last layer ---
# Plus heatmap plots (first & last layer, all heads) for one example sentence, per PDF Sec 2.3.
# With no positional embeddings, any position-correlated pattern here (e.g. distance-from-diagonal
# structure) can only come from the causal mask's visibility-count asymmetry, not from a learned
# position signal -- this is the ablation's core "what breaks" comparison point vs. the standard model.
print("\n[3/3] Attention analysis (entropy, mean distance per head; heatmaps for first & last layer)...")

def attn_entropy_per_head(w):
    """w: (num_heads, seq_len, seq_len) -> entropy per head, averaged over query positions."""
    ent = -(w * torch.log(w + 1e-10)).sum(dim=-1)  # (num_heads, seq_len)
    return ent.mean(dim=-1)  # (num_heads,)

def attn_mean_distance_per_head(w):
    """w: (num_heads, seq_len, seq_len) -> mean |query_pos - key_pos| per head, averaged over query positions."""
    num_heads, seq_len, _ = w.shape
    positions = torch.arange(seq_len, device=w.device)
    dist = (positions.view(1, -1) - positions.view(-1, 1)).abs().float()  # (seq_len, seq_len)
    weighted = (w * dist.unsqueeze(0)).sum(dim=-1)  # (num_heads, seq_len)
    return weighted.mean(dim=-1)  # (num_heads,)

ATTN_SAMPLES = 100
per_head_entropy = {"first_layer": [], "last_layer": []}
per_head_distance = {"first_layer": [], "last_layer": []}
heatmap_saved = False

for idx, text in enumerate(test_texts[:ATTN_SAMPLES]):
    if (idx + 1) % 50 == 0:
        print(f"   ...{idx + 1}/{ATTN_SAMPLES} attention samples processed")
    ids = tokenizer.encode(text[:100], add_special_tokens=False)
    if len(ids) < 2:
        continue
    input_ids = torch.tensor([ids], device=device)
    with torch.no_grad():
        _, attn_list = model(input_ids, return_attn=True)
    if not attn_list:
        continue

    first_attn, last_attn = attn_list[0], attn_list[-1]  # each: (num_heads, seq_len, seq_len)
    per_head_entropy["first_layer"].append(attn_entropy_per_head(first_attn).cpu().numpy())
    per_head_entropy["last_layer"].append(attn_entropy_per_head(last_attn).cpu().numpy())
    per_head_distance["first_layer"].append(attn_mean_distance_per_head(first_attn).cpu().numpy())
    per_head_distance["last_layer"].append(attn_mean_distance_per_head(last_attn).cpu().numpy())

    if not heatmap_saved:
        num_heads = first_attn.shape[0]
        ncols = min(4, num_heads)
        nrows = math.ceil(num_heads / ncols)
        for layer_name, attn in [("first_layer", first_attn), ("last_layer", last_attn)]:
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
            axes = np.array(axes).reshape(-1)
            for h in range(num_heads):
                im = axes[h].imshow(attn[h].detach().cpu().numpy(), cmap="viridis", aspect="auto")
                axes[h].set_title(f"Head {h}")
                axes[h].set_xlabel("Key position")
                axes[h].set_ylabel("Query position")
                fig.colorbar(im, ax=axes[h], fraction=0.046, pad=0.04)
            for h in range(num_heads, len(axes)):
                axes[h].axis("off")
            fig.suptitle(f"{LANGUAGE.capitalize()} NO-POS {layer_name.replace('_', ' ')} attention "
                         f"(example: \"{text[:40]}...\")")
            fig.tight_layout()
            heatmap_path = Path(OUT_DIR).parent / f"{LANGUAGE}_no_pos_attention_{layer_name}_heatmap.png"
            fig.savefig(heatmap_path, dpi=120)
            plt.close(fig)
            print(f"   Saved heatmap: {heatmap_path}")
        heatmap_saved = True

attn_results = {
    "first_layer": {
        "entropy_per_head": np.mean(per_head_entropy["first_layer"], axis=0).round(4).tolist() if per_head_entropy["first_layer"] else [],
        "mean_distance_per_head": np.mean(per_head_distance["first_layer"], axis=0).round(4).tolist() if per_head_distance["first_layer"] else [],
    },
    "last_layer": {
        "entropy_per_head": np.mean(per_head_entropy["last_layer"], axis=0).round(4).tolist() if per_head_entropy["last_layer"] else [],
        "mean_distance_per_head": np.mean(per_head_distance["last_layer"], axis=0).round(4).tolist() if per_head_distance["last_layer"] else [],
    },
    "num_samples_analyzed": len(per_head_entropy["first_layer"]),
}
results["attention_summary"] = attn_results
print(f"   First layer entropy/head:    {attn_results['first_layer']['entropy_per_head']}")
print(f"   Last layer entropy/head:     {attn_results['last_layer']['entropy_per_head']}")
print(f"   First layer mean dist/head:  {attn_results['first_layer']['mean_distance_per_head']}")
print(f"   Last layer mean dist/head:   {attn_results['last_layer']['mean_distance_per_head']}")

# Save results
eval_out_path = Path(OUT_DIR).parent / f"{LANGUAGE}_no_pos_phase2_evaluation.json"
with open(eval_out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\n✅ Saved evaluation metrics to {eval_out_path}")
print(f"✅ Saved attention heatmaps to {Path(OUT_DIR).parent}/{LANGUAGE}_no_pos_attention_*_heatmap.png")
print(f"\n📌 Compare against the standard model's {LANGUAGE}_phase2_evaluation.json (from pretrain_kaggle.ipynb) "
      f"and {LANGUAGE}_attention_*_heatmap.png to see what changes without positional embeddings.")
